# FedSwarm — Phase 6 main sweep (Kaggle GPU)

Person B's workstream. Runs the **576-cell main experiment** (12 strategies x 6 partition
regimes x 8 seeds), plus the client-scaling and overhead sweeps and the reproducibility check.

## Before you run anything

1. **Turn on the GPU.** Right sidebar -> Notebook options -> Accelerator -> **GPU T4 x2** (or P100).
   On CPU this will not finish.
2. **Add the dataset.** Right sidebar -> **+ Add Input** -> Datasets -> search
   `masoudnickparvar/brain-tumor-mri-dataset` -> Add.
   *You do not need Kaggle API credentials.* Adding it in the sidebar mounts it at `/kaggle/input/`.
3. **Optional: Persistence.** Right sidebar -> Notebook options -> Persistence -> *Files only*.
   This keeps `/kaggle/working` across *interactive* session restarts, which helps if your
   session times out mid-sweep. It is **not** what saves your results, and if you cannot find
   it in the UI, skip it -- see the next line for the mechanism that matters.
4. **Know how results survive: Save Version.** Results are durable because *Save Version*
   captures `/kaggle/working` as that version's **Output**, which the next session adds back
   as an input. Section 7 spells this out. Persistence is convenience; Save Version is the
   load-bearing step.

## This will not finish in one session

Kaggle caps a session at ~9 GPU-hours. The full sweep is 57,600 training rounds and will need
several sessions. That is expected and handled: every completed cell is written to its own
result file and indexed in `results/manifest.jsonl`, and re-running skips whatever is already
done. **Section 7 at the bottom tells you exactly how to resume.**

## Two gates before the sweep — do not skip them

The implementation plan (§11) is explicit that the main sweep is *not* the first thing to run:

> *"Week 5 — A1 only — go/no-go on claim C2. The week-5 gate is the important one. If A1 shows
> ACO ties random search at equal budget, stop and reframe before investing weeks 6-12."*

So this notebook runs, in order: a 20-minute mechanism check, then A1 (80 cells), and only then
the 576-cell sweep. If either gate comes back bad, **stop and talk to the team** — the sweep
answers a question the paper may no longer be able to ask.

## 1. Setup

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/researchpaper784-alt/ResearchPaper.git"
REPO_DIR = "/kaggle/working/ResearchPaper"

# BRANCH is not optional and not cosmetic. A bare `git clone` takes the repository's
# DEFAULT branch, which is `main` -- and every change this workstream needs is on the
# feature branch: the per-cell supernode count (without it overhead.yaml records K = 5..200
# while training 2 clients, and reports a flat curve as the measured O(K^2) result), the
# runners' --gpus-per-client flag, the plan's linear level set, and the Makefile's
# overridable PY/FLWR. On a `main` checkout `make ... PY=python` silently does nothing,
# because main's recipes name `.venv/bin/python` literally and Kaggle has no .venv.
BRANCH = "claude/happy-hamilton-c5jjil"

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

on = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                    capture_output=True, text=True).stdout.strip()
print("branch:", on)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)
if on != BRANCH:
    raise SystemExit(
        f"Checked out {on!r}, not {BRANCH!r}. Everything this notebook runs lives on that "
        "branch; on main the gate and the scaling sweeps are wrong rather than broken."
    )

In [ ]:
%cd /kaggle/working/ResearchPaper

# flwr[simulation] pulls in ray; installed first and on its own, same reason as the Colab
# notebook. fedswarm is installed --no-deps because its own pins (torch==2.2.2, numpy<2)
# target the Intel-macOS dev machine and do not exist for Kaggle's CUDA build -- Kaggle's
# base image already has a newer working torch/numpy.
!pip install -q "flwr[simulation]>=1.36.0,<1.37.0"
!pip install -q --no-deps -e .
!pip install -q omegaconf rich

In [ ]:
import glob
import os
import sys
from pathlib import Path

inputs = sorted(glob.glob("/kaggle/input/*"))
print("inputs mounted:", [Path(p).name for p in inputs] or "NONE")
if not inputs:
    raise SystemExit(
        "No dataset under /kaggle/input/. Fix: right sidebar -> + Add Input -> "
        "Datasets -> search masoudnickparvar/brain-tumor-mri-dataset -> Add."
    )

# Pick the input that actually holds the images, rather than trusting glob order.
# From session 2 onward there are at least TWO inputs, because section 7 tells you to add
# the previous version's output back in -- and taking inputs[0] would then point the data
# root at a results folder. `find_split_parent` would raise "Could not find a directory
# containing ['Training', 'Testing']", which reads like a corrupt dataset rather than the
# wrong input being picked.
DATA_ROOT = next(
    (p for p in inputs if any(Path(p).rglob("Training"))),
    None,
)
if DATA_ROOT is None:
    raise SystemExit(
        "None of the mounted inputs contains a Training/ directory, so none of them is "
        f"the MRI dataset. Mounted: {[Path(p).name for p in inputs]}. Add "
        "masoudnickparvar/brain-tumor-mri-dataset via + Add Input."
    )
print("dataset root:", DATA_ROOT)

# `flwr run` executes an INSTALLED COPY of the app, whose __file__ is not this clone, so the
# app resolves data/cache paths against FEDSWARM_REPO_ROOT rather than its own location.
os.environ["FEDSWARM_DATA_ROOT"] = DATA_ROOT
os.environ["FEDSWARM_REPO_ROOT"] = "/kaggle/working/ResearchPaper"
os.environ["FLWR_DISABLE_RUNTIME_DEPENDENCY_INSTALLATION"] = "1"

sys.path.insert(0, "src")
# Imported here, not at the top: fedswarm only resolves after the sys.path insert.
import torch  # noqa: E402
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible. The 576-cell sweep will not finish on CPU. Fix: right sidebar "
        "-> Notebook options -> Accelerator -> GPU T4 x2, then Run -> Restart & clear "
        "cell outputs, and re-run from cell 1."
    )
print("CPU cores:", os.cpu_count())

# Printed so that if anything below fails, this output is the whole environment report --
# paste it with the traceback rather than reconstructing it afterwards.
import flwr  # noqa: E402
from fedswarm.data.download import find_split_parent  # noqa: E402
print("flwr:", flwr.__version__, "| torch:", torch.__version__)
print("split parent:", find_split_parent(Path(DATA_ROOT)))

### Restore results from your previous session

If this is not your first session, add your **previous notebook version's output** as an input
(sidebar -> + Add Input -> Your Work -> pick the earlier version). This cell copies those results
back in so the sweep resumes instead of recomputing. Safe to run on the first session too.

In [ ]:
import shutil
from pathlib import Path

RESULTS = Path("/kaggle/working/ResearchPaper/results")
RESULTS.mkdir(parents=True, exist_ok=True)

restored = 0
for prior in glob.glob("/kaggle/input/**/results", recursive=True):
    for src in Path(prior).rglob("*.json*"):
        dst = RESULTS / src.relative_to(prior)
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
            restored += 1
print(f"restored {restored} file(s) from previous sessions")
print("completed results now present:", len(list(RESULTS.rglob('*.json'))))

## 2. Build the image cache

Decodes the JPEGs once into a memory-mapped array. I/O-bound, a few minutes.

In [ ]:
import pandas as pd

from fedswarm.data.cache import build_and_save_cache
from fedswarm.data.download import find_split_parent, resolve_root

manifest = pd.read_csv("data/processed/manifest.csv")
cache_path = build_and_save_cache(manifest, find_split_parent(resolve_root(None)), 112)
print("cache:", cache_path)

## 3. Configure the federation — not optional

`num-clients` is this project's key for how the *data* is partitioned. How many ClientApps
actually exist is Flower's `num_supernodes`, which **defaults to 2**. Left unset, every cell
would train 2 clients while recording 20.

`--client-resources-num-gpus 0.2` lets 5 ClientApps share one GPU. Raise it if you hit OOM,
lower it for more concurrency.

This cell covers section 4's `make validate-fedaco` and section 6's `main.yaml` sweep, both
of which hold K constant. **Sections 5, 6b and 6c do not need it** -- `run_sweep_granular.py`
sets the supernode count per cell from that cell's own `num-clients`, because those configs
vary it (overhead.yaml sweeps K = 5 -> 200).

In [ ]:
# GPUS_PER_CLIENT is a fraction of one card: 0.2 lets five ClientApps share it. Raise it
# if you hit CUDA OOM, lower it for more concurrency. Passed to every runner below -- at 0
# the ClientApp actors may get no GPU allocation and the sweep silently runs on CPU.
GPUS_PER_CLIENT = 0.2

# One line on purpose: a `\` continuation inside a `!` command is not reliably honoured,
# because IPython's `!` is line-oriented rather than a shell heredoc.
!flwr federation simulation-config --num-supernodes 10 --client-resources-num-cpus 1 --client-resources-num-gpus {GPUS_PER_CLIENT}

## 4. GATE 1 — does the mechanism do anything on real data? (~20 min)

Two short runs, FedACO and FedAvg on the same seed and partition, then the health check.
This is the first time FedACO will have trained on real images.

**Read all four answers before continuing.** Question 3 in particular: if the fitness optimum
is degenerate at this K, a colony that searches *well* returns "use one client, discard the
rest" — and every other signal in the report reads as success while it does.

In [ ]:
# Inlined rather than `make validate-fedaco`: `make` is not guaranteed to exist on a
# Kaggle image, and a missing binary here would look like a project failure.
GATE = (
    "num-clients=10 min-train-nodes=10 min-evaluate-nodes=10 min-available-nodes=10 "
    "num-rounds=15 local-epochs=1 regime='dirichlet' alpha=0.3 seed=0"
)
!flwr run . --stream --run-config "strategy-name='fedaco' {GATE}"
!flwr run . --stream --run-config "strategy-name='fedavg' {GATE}"
!python scripts/check_fedaco_health.py --results-dir results/fl

## 5. GATE 2 — A1, the go/no-go on claim C2 (80 cells)

The plan's week-5 gate. Replaces the colony with random search, coordinate grid search, PSO
and a GA at an **identical evaluation budget** and an identical fitness function.

> *"If ACO ties random search at equal budget, you do not have an ACO paper — you have an
> adaptive-weighting paper, and you should reframe honestly rather than overclaim."*

The risk register rates this **Medium-High likelihood, fatal to the framing.** If ACO does not
separate from the controls here, **stop and tell the team before running section 6.**

In [ ]:
!python scripts/run_sweep_granular.py --config configs/experiment/ablation_a1.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
# Read the verdict. `aco` must beat the four controls -- and by more than seed noise.
!python scripts/make_tables.py --results-dir results/fl --out paper/tables --expected-seeds 8

## 6. The main sweep — 576 cells

Only run this if both gates above came back clean.

Resumable: already-completed cells are skipped. If the session dies, save the version and
re-run from the top next time. `--dry-run` first to see the projection against a real
per-round cost now that a GPU has measured one.

In [ ]:
!python scripts/run_sweep.py --config configs/experiment/main.yaml --dry-run

In [ ]:
!python scripts/run_sweep.py --config configs/experiment/main.yaml --gpus-per-client {GPUS_PER_CLIENT}

## 6b. Client scaling and overhead (B's other two sweeps)

In [ ]:
!python scripts/run_sweep_granular.py --config configs/experiment/main_client_scale.yaml --gpus-per-client {GPUS_PER_CLIENT}
!python scripts/run_sweep_granular.py --config configs/experiment/overhead.yaml --gpus-per-client {GPUS_PER_CLIENT}

## 6c. Reproducibility check (Phase 10)

Runs the smoke config and checks final metrics land inside the recorded tolerance band.

In [ ]:
!python scripts/verify_repro.py

## 7. Before the session ends — SAVE, or you lose everything

1. **Save Version** (top right). *Quick Save* commits the current `/kaggle/working` without
   re-running; *Save & Run All* re-executes the notebook from scratch in a batch session --
   for a resumed sweep you almost always want **Quick Save**, since Save & Run All restarts
   from cell 1 in a fresh container. Files under `/kaggle/working` become the version's
   Output; anything outside it is discarded.
   
   This is the step that makes results durable. The Persistence setting does not replace it.
2. Next session: **+ Add Input -> Your Work -> this notebook's previous version**, then run from
   the top. Section 1's restore cell picks the results back up and the sweep continues.

The cell below shows what has finished, so you know where you are.

In [ ]:
import glob

!python scripts/run_sweep.py --config configs/experiment/main.yaml --dry-run
print("result files on disk:", len(glob.glob("/kaggle/working/ResearchPaper/results/**/*.json", recursive=True)))